# Exploratory Data Analysis

Sits between `01_data_ingestion` and the feature tables it justifies (`02_feature_store`,
`data_pipelines/11_features_nightly`). Nothing here writes anything — read-only, against
bronze (`mlo.weather_mlops.noaa_historical_daily`) plus the already-built feature/label
tables, to check the assumptions those notebooks already made:

- next-day `PRCP > 0.5mm` as the label (`MODEL_CARD.md`)
- lagged/rolling precipitation as v2 features (`02_feature_store.ipynb`)
- the other nine stations as v3 features, on the theory that Midwest systems track
  west-to-east (`data_pipelines/README.md`)

Re-run this after any change to `DEFAULT_STATIONS` / `DEFAULT_DATATYPES` in
`data_pipelines/noaa_client.py` — the numbers below are only as current as bronze.

In [ ]:
import os
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Files in Repos puts this notebook's own directory on sys.path automatically, but running
# from a plain checkout doesn't -- add the repo root so `from data_pipelines...` resolves
# either way, matching 01_data_ingestion.ipynb.
REPO_ROOT = os.getcwd()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

pd.set_option("display.max_columns", None)

CATALOG = "mlo"
PRIMARY_STATION = "GHCND:USW00014819"  # Chicago Midway -- the prediction target
NOAA_TABLE = f"{CATALOG}.weather_mlops.noaa_historical_daily"
LABELS_TABLE = f"{CATALOG}.features.weather_labels"
V3_TABLE = f"{CATALOG}.features.weather_daily_v3"

bronze = spark.table(NOAA_TABLE).toPandas()
bronze["date"] = pd.to_datetime(bronze["date"])
print(f"{len(bronze)} station-days | {bronze['station'].nunique()} stations | "
      f"{bronze['date'].min().date()} -> {bronze['date'].max().date()}")

## Coverage by station

Bronze is long in the station dimension -- one row per `(station, date)`. Before trusting
anything downstream, check every station actually has the date range and volume the nightly
`MERGE` assumes, and confirm Midway (the target) isn't the outlier.

In [ ]:
coverage = (bronze.groupby("station")
            .agg(days=("date", "size"),
                 first_date=("date", "min"),
                 last_date=("date", "max"))
            .sort_values("days"))
coverage["is_primary"] = coverage.index == PRIMARY_STATION
coverage

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
colors = ["#d62728" if s == PRIMARY_STATION else "#1f77b4" for s in coverage.index]
ax.barh(coverage.index, coverage["days"], color=colors)
ax.set_xlabel("station-days in bronze")
ax.set_title("Row count per station (red = primary/Midway)")
plt.tight_layout()
plt.show()

## Missingness by station x datatype

`WT01`/`WT02`/`WT03` are event flags, not measurements -- NOAA only writes a value on days
the phenomenon occurred, so high null rates there are expected, not a data-quality problem
(see the comment in `01_data_ingestion.ipynb` and `noaa_client.py`). Everything else should be
consistently well-populated across stations; a station that isn't is a candidate to drop from
`DEFAULT_STATIONS` before it pollutes v3.

In [ ]:
DATATYPES = ["AWND", "PRCP", "TMAX", "TMIN", "WDF2", "WDF5", "WSF2", "WSF5",
             "SNWD", "WT01", "WT02", "WT03"]
present = [c for c in DATATYPES if c in bronze.columns]

miss = bronze.groupby("station")[present].apply(lambda g: g.isna().mean().round(3))
miss

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(miss.values, cmap="Reds", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(present))); ax.set_xticklabels(present, rotation=45, ha="right")
ax.set_yticks(range(len(miss))); ax.set_yticklabels(miss.index)
ax.set_title("Null rate by station x datatype")
fig.colorbar(im, ax=ax, label="fraction null")
plt.tight_layout()
plt.show()

## Distributions -- Midway

Everything from here on filters to `PRIMARY_STATION`, matching the scope of v1/v2 features,
the labels, and the baseline model. `PRCP` is the one that matters most: it's both the
label's source column and famously zero-inflated, which is why a mean/threshold summary
undersells it relative to the histogram.

In [ ]:
midway = bronze[bronze["station"] == PRIMARY_STATION].sort_values("date").reset_index(drop=True)

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for ax, col in zip(axes.flat, ["TMAX", "TMIN", "AWND", "PRCP"]):
    ax.hist(midway[col].dropna(), bins=40, color="#1f77b4")
    ax.set_title(col)
plt.tight_layout()
plt.show()

zero_prcp = (midway["PRCP"] == 0).mean()
print(f"PRCP == 0 on {zero_prcp:.1%} of days -- zero-inflated, as expected for daily precip")
print(f"PRCP > 0.5mm (the label threshold) on {(midway['PRCP'] > 0.5).mean():.1%} of days")

## Seasonality

Chicago precipitation and temperature both have a strong annual cycle. This matters for
modeling because a next-day model with no seasonal feature is implicitly relying on
`TMAX`/`TMIN`/lagged `PRCP` to encode "what season is it" -- worth knowing when reading the
baseline's near-zero coefficients in `MODEL_CARD.md`.

In [ ]:
midway["month"] = midway["date"].dt.month
monthly = midway.groupby("month")[["TMAX", "TMIN", "PRCP"]].mean()

fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.plot(monthly.index, monthly["TMAX"], marker="o", label="TMAX (mean)", color="#d62728")
ax1.plot(monthly.index, monthly["TMIN"], marker="o", label="TMIN (mean)", color="#1f77b4")
ax1.set_xlabel("month"); ax1.set_ylabel("temp")
ax1.set_xticks(range(1, 13))
ax2 = ax1.twinx()
ax2.bar(monthly.index, monthly["PRCP"], alpha=0.25, color="#2ca02c", label="PRCP (mean)")
ax2.set_ylabel("precip (mm)")
fig.legend(loc="upper center", bbox_to_anchor=(0.5, 1.05), ncol=3)
ax1.set_title("Monthly climatology -- Midway")
plt.tight_layout()
plt.show()

## Does rain persist? (justifies `PRCP_lag1` / `PRCP_roll3`)

v2 added lagged and rolling precipitation because same-day temperature/wind alone showed
almost no next-day signal (baseline ROC-AUC 0.527, `MODEL_CARD.md`). Before taking that
feature choice on faith, check it directly: is a rainy day more likely to be followed by
another rainy day than base rate would predict?

In [ ]:
is_wet = (midway["PRCP"] > 0.5).astype(int)
base_rate = is_wet.mean()

lags = range(1, 6)
lift = {}
for L in lags:
    lagged = is_wet.shift(L)
    # P(wet today | wet L days ago) vs base rate
    p_wet_given_wet = is_wet[lagged == 1].mean()
    lift[L] = p_wet_given_wet / base_rate

print(f"base rate of wet days: {base_rate:.1%}")
for L, v in lift.items():
    print(f"  P(wet | wet {L}d ago) is {v:.2f}x base rate")

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(list(lift.keys()), list(lift.values()), color="#1f77b4")
ax.axhline(1.0, color="gray", linestyle="--", label="base rate")
ax.set_xlabel("lag (days)"); ax.set_ylabel("P(wet | wet at lag) / base rate")
ax.set_title("Precipitation persistence at Midway")
ax.legend()
plt.tight_layout()
plt.show()

## Does upstream weather lead Midway? (justifies v3's wide, multi-station shape)

`data_pipelines/README.md` frames v3 around Midwest systems tracking west-to-east, so a
station like Rockford (`RFD`, west/north-west of Midway) should show correlation with
Midway's weather *today* that's stronger looking at Rockford *yesterday* than Rockford
*today* -- system arrives here after it arrives there. Pivot bronze to one row per date to
check this directly rather than assuming the geography argument holds.

In [ ]:
from data_pipelines.noaa_client import DEFAULT_STATIONS, make_headers, get_station_names

# Real NOAA station names rather than hand-written short labels -- one GET per station
# (see get_station_names in data_pipelines/noaa_client.py), so this is a handful of calls
# for DEFAULT_STATIONS' 11 stations.
NOAA_TOKEN = dbutils.secrets.get(scope="mlo", key="WEATHER_API_KEY")
STATION_NAMES = get_station_names(DEFAULT_STATIONS, make_headers(NOAA_TOKEN))
STATION_NAMES

prcp_wide = bronze.pivot(index="date", columns="station", values="PRCP").sort_index()
prcp_wide = prcp_wide.rename(columns=STATION_NAMES)
target_name = STATION_NAMES[PRIMARY_STATION]
target = prcp_wide[target_name]

same_day_corr = prcp_wide.corr()[target_name].drop(target_name)
lag1_corr = prcp_wide.drop(columns=target_name).shift(1).corrwith(target)

compare = pd.DataFrame({"same_day": same_day_corr, "lag1_leads_midway": lag1_corr}).sort_values(
    "lag1_leads_midway", ascending=False)
compare

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
idx = np.arange(len(compare))
width = 0.35
ax.bar(idx - width / 2, compare["same_day"], width, label="same-day corr with MDW", color="#1f77b4")
ax.bar(idx + width / 2, compare["lag1_leads_midway"], width, label="station(t-1) vs MDW(t)", color="#d62728")
ax.set_xticks(idx); ax.set_xticklabels(compare.index, rotation=45, ha="right")
ax.axhline(0, color="gray", linewidth=0.8)
ax.set_ylabel("correlation with Midway PRCP")
ax.set_title("Same-day vs. lagged correlation with Midway precipitation")
ax.legend()
plt.tight_layout()
plt.show()

stronger_lagged = (compare["lag1_leads_midway"] > compare["same_day"]).sum()
print(f"{stronger_lagged}/{len(compare)} upstream stations correlate more strongly with "
      f"Midway lagged by a day than same-day -- consistent with the west-to-east framing "
      f"v3 is built on.")

## Feature correlation with the label -- v3

Read `weather_daily_v3` joined to `weather_labels` and rank every engineered feature by
correlation with `Bad` (tomorrow's rain at Midway). This is the same table `05_automl.ipynb`
trains on -- useful as a sanity check before trusting a model's feature importances, and as a
first read on whether the extra upstream-station columns are actually carrying signal or just
adding the ~44-features-for-~475-events sparsity `data_pipelines/README.md` already flags.

In [ ]:
v3 = spark.table(V3_TABLE).toPandas()
labels = spark.table(LABELS_TABLE).toPandas()
labels = labels[labels["station"] == PRIMARY_STATION]

merged = v3.merge(labels[["date", "Bad"]], on="date", how="inner")
feature_cols = [c for c in merged.columns if c not in ("station", "date", "Bad")]

corr_with_label = (merged[feature_cols + ["Bad"]].corr(numeric_only=True)["Bad"]
                    .drop("Bad").sort_values(key=np.abs, ascending=False))

print(f"{len(feature_cols)} features x {len(merged)} labeled rows "
      f"({merged['Bad'].mean():.1%} positive class)")
corr_with_label.head(20)

In [ ]:
top = corr_with_label.head(20)
fig, ax = plt.subplots(figsize=(7, 7))
colors = ["#d62728" if v < 0 else "#1f77b4" for v in top.values]
ax.barh(top.index[::-1], top.values[::-1], color=colors[::-1])
ax.axvline(0, color="gray", linewidth=0.8)
ax.set_xlabel("correlation with Bad (tomorrow's rain)")
ax.set_title("Top 20 v3 features by |correlation| with label")
plt.tight_layout()
plt.show()

## Summary

- **Coverage**: all ten stations should show comparable row counts and date ranges to
  Midway (see the bar chart above) -- if one doesn't, that's a `DEFAULT_STATIONS` fix, not a
  modeling problem.
- **Missingness**: `WT01`-`WT03` nulls are structural (event-not-occurred), not
  data-quality issues -- confirmed here, matching the existing comment in
  `01_data_ingestion.ipynb`.
- **PRCP is zero-inflated**, which is exactly why a threshold label (`PRCP > 0.5mm`) rather
  than a regression target was the right call for a first model.
- **Rain persists** day-to-day at Midway well above base rate, which is the direct
  empirical justification for `PRCP_lag1` / `PRCP_roll3` in v2 -- not just a modeling
  convention.
- **Upstream stations lead, not coincide**: several show a stronger lag-1 correlation with
  Midway than same-day, supporting v3's west-to-east framing and the choice to pivot other
  stations into lagged wide columns rather than treating them as same-day covariates.
- **Feature correlation with the label** is a useful pre-check on `05_automl.ipynb`'s
  results -- features that show up with near-zero correlation here and high importance in a
  tree model are worth a second look before trusting them, given how few positive events v3
  has to work with per feature.

Re-run whenever `DEFAULT_STATIONS`, `DEFAULT_DATATYPES`, or the v3 `FEATURE_SPEC` change --
the numbers above are a snapshot of bronze/v3 as they exist right now, not a permanent
result.